In [338]:
reference_table_names = ['bill_id', 'bill_id_dup', 'vendor_id', 'vendor_name',
       'bni_created_time', 'delay_days', 'shipped_dt', 'eta_dt', 'po_date_dt',
       'receipt_dt', 'shipment_days', 'promised_transit_days',
       'days_until_eta', 'days_since_ship_so_far', 'lead_time_days', 'coo',
       'scac', 'tariff_amount', 'ocean_freight', 'delivery_terms',
       'po_shipment_terms', 'tariff_type', 'total_bcy', 'quantity_in',
       'item_sku', 'item_brand', 'item_manufacturer', 'item_product_category',
       'item_size', 'vendor_avg_promised_transit_days',
       'vendor_p50_promised_transit_days', 'vendor_p90_promised_transit_days',
       'vendor_avg_realized_delay_days', 'vendor_p50_realized_delay_days',
       'vendor_p90_realized_delay_days', 'vendor_on_time_rate',
       'vendor_shipments_with_receipt']

In [339]:
type(reference_table_names)

list

In [340]:
from dotenv import load_dotenv
import os
import clickhouse_connect
import pandas as pd
from datetime import datetime


load_dotenv()

client = clickhouse_connect.get_client(
    host=os.getenv('CLICKHOUSE_HOST'),
    port=int(os.getenv('CLICKHOUSE_PORT', 8123)), 
    username=os.getenv('CLICKHOUSE_USERNAME'),
    password=os.getenv('CLICKHOUSE_PASSWORD'),
    database=os.getenv('CLICKHOUSE_DATABASE')
)

In [341]:
shipment_dataset_query_1 = """
/* Row-level shipment features only (no vendor aggregates) */
SELECT
    -- Primary keys / bookkeeping
    bni.bill_id                                         AS bill_id,
    b.bill_id                                           AS bill_id_dup,
    v.vendor_id                                         AS vendor_id,
    v.vendor_name                                       AS vendor_name,
    bni.created_time                                    AS bni_created_time,

    -- 📦 Core dates (parsed) — receipt_date is intentionally NOT used for row-level features
    round(greatest(
        dateDiff('day', 
            toDate(parseDateTimeBestEffortOrNull(b.eta)), 
            toDate(parseDateTimeBestEffortOrNull(b.receipt_date))
        ), 0
    ), 2)                                               AS delay_days,
    parseDateTimeBestEffortOrNull(b.shipped_date)       AS shipped_dt,
    parseDateTimeBestEffortOrNull(b.eta)                AS eta_dt,
    parseDateTimeBestEffortOrNull(p.purchase_order_date) AS po_date_dt,
    parseDateTimeBestEffortOrNull(b.receipt_date)       AS receipt_dt,

    -- ⏩ Scoring-safe timing features (all available at ship-time / scoring-time)
    toFloat64(dateDiff('day',
        toDate(shipped_dt),
        toDate(ifNull(receipt_dt, eta_dt))              -- receipt if present, else ETA
    ))                                                  AS shipment_days,
    toFloat64(dateDiff('day', toDate(shipped_dt), toDate(eta_dt))) AS promised_transit_days,
    toFloat64(dateDiff('day', toDate(now()), toDate(eta_dt)))      AS days_until_eta,
    toFloat64(dateDiff('day', toDate(shipped_dt), toDate(now())))  AS days_since_ship_so_far,
    toFloat64(dateDiff('day', toDate(po_date_dt), toDate(shipped_dt))) AS lead_time_days,

    -- 🌍 Shipping details (static/manifest fields)
    b.coo                                               AS coo,
    b.scac                                              AS scac,
    round(toFloat64OrNull(b.tariff_amount), 2)          AS tariff_amount,
    round(toFloat64OrNull(b.ocean_freight), 2)          AS ocean_freight,
    p.delivery_terms                                    AS delivery_terms,
    p.shipment_terms                                    AS po_shipment_terms,
    p.tariff_type                                       AS tariff_type,

    -- 💰 Cost / product / vendor info
    p.total_bcy                                         AS total_bcy,
    round(toFloat64OrNull(bni.quantity_in), 2)          AS quantity_in,
    i.sku                                               AS item_sku,
    i.brand                                             AS item_brand,
    i.manufacturer                                      AS item_manufacturer,
    i.product_category                                  AS item_product_category,
    i.size                                              AS item_size

FROM zoho_books_analytics.batch_number_in AS bni
INNER JOIN zoho_books_analytics.bills AS b
    ON bni.bill_id = b.bill_id
INNER JOIN zoho_books_analytics.bill_item AS bi
    ON b.bill_id = bi.bill_id
INNER JOIN zoho_books_analytics.purchase_orders AS p
    ON b.purchase_order = p.purchase_order_number
INNER JOIN zoho_books_analytics.items AS i
    ON bi.product_id = i.item_id
INNER JOIN zoho_books_analytics.sales_orders AS so
    ON p.reference_number = so.sales_order
INNER JOIN zoho_books_analytics.customers AS c
    ON c.customer_id = so.customer_id
INNER JOIN zoho_books_analytics.customer_item_mapping AS ci
    ON i.sku = ci.az_sku
INNER JOIN zoho_books_analytics.vendors AS v
    ON v.vendor_id = b.vendor_id

WHERE
    c.customer_name LIKE 'Walmart%'
    AND b.shipped_date IS NOT NULL
    AND b.eta IS NOT NULL
ORDER BY
    bni.created_time DESC
"""

shipment_dataset_query_2 = """
/* Safe per-bill vendor aggregates keyed by (bill_id, vendor_id) */
SELECT
    b_curr.bill_id AS bill_id,
    v_curr.vendor_id AS vendor_id,

    -- Promised transit (ship -> eta) historical metrics
    round(AVG(toFloat64(dateDiff('day',
        toDate(parseDateTimeBestEffortOrNull(b_hist.shipped_date)),
        toDate(parseDateTimeBestEffortOrNull(b_hist.eta))
    ))), 2) AS vendor_avg_promised_transit_days,

    round(quantileExact(0.5)(toFloat64(dateDiff('day',
        toDate(parseDateTimeBestEffortOrNull(b_hist.shipped_date)),
        toDate(parseDateTimeBestEffortOrNull(b_hist.eta))
    ))), 2) AS vendor_p50_promised_transit_days,

    round(quantileExact(0.9)(toFloat64(dateDiff('day',
        toDate(parseDateTimeBestEffortOrNull(b_hist.shipped_date)),
        toDate(parseDateTimeBestEffortOrNull(b_hist.eta))
    ))), 2) AS vendor_p90_promised_transit_days,

    -- Realized delay (eta -> receipt) historical metrics (non-negative)
    round(AVG(toFloat64(greatest(
        dateDiff('day',
            toDate(parseDateTimeBestEffortOrNull(b_hist.eta)),
            toDate(parseDateTimeBestEffortOrNull(b_hist.receipt_date))
        ), 0
    ))), 2) AS vendor_avg_realized_delay_days,

    round(quantileExact(0.5)(toFloat64(greatest(
        dateDiff('day',
            toDate(parseDateTimeBestEffortOrNull(b_hist.eta)),
            toDate(parseDateTimeBestEffortOrNull(b_hist.receipt_date))
        ), 0
    ))), 2) AS vendor_p50_realized_delay_days,

    round(quantileExact(0.9)(toFloat64(greatest(
        dateDiff('day',
            toDate(parseDateTimeBestEffortOrNull(b_hist.eta)),
            toDate(parseDateTimeBestEffortOrNull(b_hist.receipt_date))
        ), 0
    ))), 2) AS vendor_p90_realized_delay_days,

    round(AVG(toUInt8(greatest(
        dateDiff('day',
            toDate(parseDateTimeBestEffortOrNull(b_hist.eta)),
            toDate(parseDateTimeBestEffortOrNull(b_hist.receipt_date))
        ), 0) = 0
    )), 4) AS vendor_on_time_rate,

    COUNT() AS vendor_shipments_with_receipt

FROM zoho_books_analytics.bills AS b_curr
INNER JOIN zoho_books_analytics.vendors AS v_curr
    ON v_curr.vendor_id = b_curr.vendor_id

/* Historical shipments for the same vendor, strictly before current shipped date, within 365 days */
INNER JOIN zoho_books_analytics.bills AS b_hist
    ON b_hist.vendor_id = v_curr.vendor_id
   AND b_hist.shipped_date IS NOT NULL
   AND b_hist.eta IS NOT NULL
   AND b_hist.receipt_date IS NOT NULL
   AND b_hist.receipt_date != ''
   AND toDate(parseDateTimeBestEffortOrNull(b_hist.shipped_date))
       BETWEEN addDays(toDate(parseDateTimeBestEffortOrNull(b_curr.shipped_date)), -365)
           AND addDays(toDate(parseDateTimeBestEffortOrNull(b_curr.shipped_date)), -1)

/* Keep the same Walmart scope as the main query */
INNER JOIN zoho_books_analytics.purchase_orders AS p_hist
    ON b_hist.purchase_order = p_hist.purchase_order_number
INNER JOIN zoho_books_analytics.sales_orders AS so_hist
    ON p_hist.reference_number = so_hist.sales_order
INNER JOIN zoho_books_analytics.customers AS c_hist
    ON c_hist.customer_id = so_hist.customer_id

WHERE
    c_hist.customer_name LIKE 'Walmart%'
    AND b_curr.shipped_date IS NOT NULL
    AND b_curr.eta IS NOT NULL

GROUP BY
    b_curr.bill_id, v_curr.vendor_id
"""


In [342]:
# --- Query 1: base features (unchanged) ---
res1 = client.query(shipment_dataset_query_1)
df1 = pd.DataFrame(res1.result_rows, columns=list(res1.column_names))

# --- Query 2: vendor aggregates (may return bill_id_key; make sure to rename) ---
res2 = client.query(shipment_dataset_query_2)
df2 = pd.DataFrame(res2.result_rows, columns=list(res2.column_names))


In [343]:
df1.columns

Index(['bill_id', 'bill_id_dup', 'vendor_id', 'vendor_name',
       'bni_created_time', 'delay_days', 'shipped_dt', 'eta_dt', 'po_date_dt',
       'receipt_dt', 'shipment_days', 'promised_transit_days',
       'days_until_eta', 'days_since_ship_so_far', 'lead_time_days', 'coo',
       'scac', 'tariff_amount', 'ocean_freight', 'delivery_terms',
       'po_shipment_terms', 'tariff_type', 'total_bcy', 'quantity_in',
       'item_sku', 'item_brand', 'item_manufacturer', 'item_product_category',
       'item_size'],
      dtype='object')

In [344]:
df2.columns

Index(['bill_id', 'vendor_id', 'vendor_avg_promised_transit_days',
       'vendor_p50_promised_transit_days', 'vendor_p90_promised_transit_days',
       'vendor_avg_realized_delay_days', 'vendor_p50_realized_delay_days',
       'vendor_p90_realized_delay_days', 'vendor_on_time_rate',
       'vendor_shipments_with_receipt'],
      dtype='object')

In [345]:
shipment_dataset_df = df1.merge(df2, on=["bill_id", "vendor_id"], how="left")

In [346]:
shipment_dataset_df.columns == reference_table_names

array([ True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True])

In [347]:
df1.shape, df2.shape, shipment_dataset_df.shape

((2424, 29), (6427, 10), (2424, 37))

In [348]:
shipment_dataset_df.drop_duplicates(inplace=True)

In [349]:
shipment_dataset_df.dropna(subset=['receipt_dt'], inplace=True)

In [350]:
shipment_dataset_df.columns.to_list() == reference_table_names

True

In [351]:
shipment_dataset_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 847 entries, 52 to 2423
Data columns (total 37 columns):
 #   Column                            Non-Null Count  Dtype                  
---  ------                            --------------  -----                  
 0   bill_id                           847 non-null    object                 
 1   bill_id_dup                       847 non-null    object                 
 2   vendor_id                         847 non-null    object                 
 3   vendor_name                       847 non-null    object                 
 4   bni_created_time                  847 non-null    datetime64[ns, Etc/UTC]
 5   delay_days                        847 non-null    int64                  
 6   shipped_dt                        847 non-null    datetime64[ns, Etc/UTC]
 7   eta_dt                            847 non-null    datetime64[ns, Etc/UTC]
 8   po_date_dt                        847 non-null    datetime64[ns, Etc/UTC]
 9   receipt_dt              

## Inference level query breakdown

In [352]:
inference_data_query_1 = """
/* Row-level shipment features only (no vendor aggregates) */
SELECT
    -- Primary keys / bookkeeping
    bni.bill_id                                         AS bill_id,
    b.bill_id                                           AS bill_id_dup,
    v.vendor_id                                         AS vendor_id,
    v.vendor_name                                       AS vendor_name,
    bni.created_time                                    AS bni_created_time,

    -- 📦 Core dates (parsed) — receipt_date is intentionally NOT used for row-level features
    round(GREATEST(
        dateDiff('day', 
            toDate(parseDateTimeBestEffortOrNull(b.eta)), 
            toDate(parseDateTimeBestEffortOrNull(b.receipt_date))
        ), 
    0), 2)                                              AS delay_days,
    parseDateTimeBestEffortOrNull(b.shipped_date)       AS shipped_dt,
    parseDateTimeBestEffortOrNull(b.eta)                AS eta_dt,
    parseDateTimeBestEffortOrNull(p.purchase_order_date) AS po_date_dt,
    parseDateTimeBestEffortOrNull(b.receipt_date)       AS receipt_dt,

    -- ⏩ Scoring-safe timing features
    toFloat64(dateDiff('day',
        toDate(shipped_dt),
        toDate(ifNull(receipt_dt, eta_dt))
    ))                                                  AS shipment_days,
    toFloat64(dateDiff('day', toDate(shipped_dt), toDate(eta_dt)))      AS promised_transit_days,
    toFloat64(dateDiff('day', toDate(now()), toDate(eta_dt)))           AS days_until_eta,
    toFloat64(dateDiff('day', toDate(shipped_dt), toDate(now())))       AS days_since_ship_so_far,
    toFloat64(dateDiff('day', toDate(po_date_dt), toDate(shipped_dt)))  AS lead_time_days,

    -- 🌍 Shipping details (static/manifest fields)
    b.coo                                               AS coo,
    b.scac                                              AS scac,
    round(toFloat64OrNull(b.tariff_amount), 2)          AS tariff_amount,
    round(toFloat64OrNull(b.ocean_freight), 2)          AS ocean_freight,
    p.delivery_terms                                    AS delivery_terms,
    p.shipment_terms                                    AS po_shipment_terms,
    p.tariff_type                                       AS tariff_type,

    -- 💰 Cost / product / vendor info
    p.total_bcy                                         AS total_bcy,
    round(toFloat64OrNull(bni.quantity_in), 2)          AS quantity_in,
    i.sku                                               AS item_sku,
    i.brand                                             AS item_brand,
    i.manufacturer                                      AS item_manufacturer,
    i.product_category                                  AS item_product_category,
    i.size                                              AS item_size

FROM zoho_books_analytics.batch_number_in AS bni
INNER JOIN zoho_books_analytics.bills AS b
    ON bni.bill_id = b.bill_id
INNER JOIN zoho_books_analytics.bill_item AS bi
    ON b.bill_id = bi.bill_id
INNER JOIN zoho_books_analytics.purchase_orders AS p
    ON b.purchase_order = p.purchase_order_number
INNER JOIN zoho_books_analytics.items AS i
    ON bi.product_id = i.item_id
INNER JOIN zoho_books_analytics.sales_orders AS so
    ON p.reference_number = so.sales_order
INNER JOIN zoho_books_analytics.customers AS c
    ON c.customer_id = so.customer_id
INNER JOIN zoho_books_analytics.customer_item_mapping AS ci
    ON i.sku = ci.az_sku
INNER JOIN zoho_books_analytics.vendors AS v
    ON v.vendor_id = b.vendor_id

WHERE
    c.customer_name LIKE 'Walmart%'
    AND bni.created_time BETWEEN '2025-08-01' AND '2025-10-01'

ORDER BY bni.created_time DESC
"""


In [353]:
inference_data_query_2 = """
/* Safe per-bill vendor aggregates keyed by (bill_id, vendor_id) */
SELECT
    b_curr.bill_id AS bill_id,
    v_curr.vendor_id AS vendor_id,

    -- Promised transit (ship -> eta) historical metrics
    round(AVG(toFloat64(dateDiff('day',
        toDate(parseDateTimeBestEffortOrNull(b_hist.shipped_date)),
        toDate(parseDateTimeBestEffortOrNull(b_hist.eta))
    ))), 2) AS vendor_avg_promised_transit_days,

    round(quantileExact(0.5)(toFloat64(dateDiff('day',
        toDate(parseDateTimeBestEffortOrNull(b_hist.shipped_date)),
        toDate(parseDateTimeBestEffortOrNull(b_hist.eta))
    ))), 2) AS vendor_p50_promised_transit_days,

    round(quantileExact(0.9)(toFloat64(dateDiff('day',
        toDate(parseDateTimeBestEffortOrNull(b_hist.shipped_date)),
        toDate(parseDateTimeBestEffortOrNull(b_hist.eta))
    ))), 2) AS vendor_p90_promised_transit_days,

    -- Realized delay (eta -> receipt) historical metrics (non-negative)
    round(AVG(toFloat64(GREATEST(
        dateDiff('day',
            toDate(parseDateTimeBestEffortOrNull(b_hist.eta)),
            toDate(parseDateTimeBestEffortOrNull(b_hist.receipt_date))
        ), 0
    ))), 2) AS vendor_avg_realized_delay_days,

    round(quantileExact(0.5)(toFloat64(GREATEST(
        dateDiff('day',
            toDate(parseDateTimeBestEffortOrNull(b_hist.eta)),
            toDate(parseDateTimeBestEffortOrNull(b_hist.receipt_date))
        ), 0
    ))), 2) AS vendor_p50_realized_delay_days,

    round(quantileExact(0.9)(toFloat64(GREATEST(
        dateDiff('day',
            toDate(parseDateTimeBestEffortOrNull(b_hist.eta)),
            toDate(parseDateTimeBestEffortOrNull(b_hist.receipt_date))
        ), 0
    ))), 2) AS vendor_p90_realized_delay_days,

    round(AVG(toUInt8(GREATEST(
        dateDiff('day',
            toDate(parseDateTimeBestEffortOrNull(b_hist.eta)),
            toDate(parseDateTimeBestEffortOrNull(b_hist.receipt_date))
        ), 0) = 0
    )), 4) AS vendor_on_time_rate,

    COUNT() AS vendor_shipments_with_receipt

FROM zoho_books_analytics.bills AS b_curr
INNER JOIN zoho_books_analytics.vendors AS v_curr
    ON v_curr.vendor_id = b_curr.vendor_id

/* Historical shipments for the same vendor, strictly before current shipped date, within 365 days */
INNER JOIN zoho_books_analytics.bills AS b_hist
    ON b_hist.vendor_id = v_curr.vendor_id
   AND b_hist.shipped_date IS NOT NULL
   AND b_hist.eta IS NOT NULL
   AND b_hist.receipt_date IS NOT NULL
   AND b_hist.receipt_date != ''
   AND toDate(parseDateTimeBestEffortOrNull(b_hist.shipped_date))
       BETWEEN addDays(toDate(parseDateTimeBestEffortOrNull(b_curr.shipped_date)), -365)
           AND addDays(toDate(parseDateTimeBestEffortOrNull(b_curr.shipped_date)), -1)

/* Keep the same Walmart scope as the main query */
INNER JOIN zoho_books_analytics.purchase_orders AS p_hist
    ON b_hist.purchase_order = p_hist.purchase_order_number
INNER JOIN zoho_books_analytics.sales_orders AS so_hist
    ON p_hist.reference_number = so_hist.sales_order
INNER JOIN zoho_books_analytics.customers AS c_hist
    ON c_hist.customer_id = so_hist.customer_id

WHERE
    c_hist.customer_name LIKE 'Walmart%'
    AND b_curr.shipped_date IS NOT NULL
    AND b_curr.eta IS NOT NULL

GROUP BY
    b_curr.bill_id, v_curr.vendor_id
"""


In [354]:
# Query 1: base features in the inference window
res1 = client.query(inference_data_query_1)
df1 = pd.DataFrame(res1.result_rows, columns=list(res1.column_names))

# Query 2: leakage-safe vendor aggregates
res2 = client.query(inference_data_query_2)
df2 = pd.DataFrame(res2.result_rows, columns=list(res2.column_names))

# Merge (LEFT join to preserve all inference rows), sort by latest created_time
inference_data_df = (
    df1.merge(df2, how="left", on=["bill_id", "vendor_id"])
       .sort_values("bni_created_time", ascending=False)
       .reset_index(drop=True)
)


In [355]:
inference_reference_query = ['bill_id', 'bill_id_dup', 'vendor_id', 'vendor_name',
       'bni_created_time', 'delay_days', 'shipped_dt', 'eta_dt', 'po_date_dt',
       'receipt_dt', 'shipment_days', 'promised_transit_days',
       'days_until_eta', 'days_since_ship_so_far', 'lead_time_days', 'coo',
       'scac', 'tariff_amount', 'ocean_freight', 'delivery_terms',
       'po_shipment_terms', 'tariff_type', 'total_bcy', 'quantity_in',
       'item_sku', 'item_brand', 'item_manufacturer', 'item_product_category',
       'item_size', 'vendor_avg_promised_transit_days',
       'vendor_p50_promised_transit_days', 'vendor_p90_promised_transit_days',
       'vendor_avg_realized_delay_days', 'vendor_p50_realized_delay_days',
       'vendor_p90_realized_delay_days', 'vendor_on_time_rate',
       'vendor_shipments_with_receipt']

In [356]:
inference_data_df.columns.to_list() == inference_reference_query

True

In [357]:
inference_data_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 377 entries, 0 to 376
Data columns (total 37 columns):
 #   Column                            Non-Null Count  Dtype                  
---  ------                            --------------  -----                  
 0   bill_id                           377 non-null    object                 
 1   bill_id_dup                       377 non-null    object                 
 2   vendor_id                         377 non-null    object                 
 3   vendor_name                       377 non-null    object                 
 4   bni_created_time                  377 non-null    datetime64[ns, Etc/UTC]
 5   delay_days                        377 non-null    int64                  
 6   shipped_dt                        377 non-null    datetime64[ns, Etc/UTC]
 7   eta_dt                            377 non-null    datetime64[ns, Etc/UTC]
 8   po_date_dt                        377 non-null    datetime64[ns, Etc/UTC]
 9   receipt_dt           

## Testing with the cube Server

In [358]:
import os
import requests
from dotenv import load_dotenv

load_dotenv()


class CubeJSClient:
    def __init__(self, token: str = None, base_url: str = None):
        self.base_url = base_url or os.getenv("CUBEJS_BASE_URL")
        self.access_token = token or os.getenv("CUBEJS_TOKEN")

    def get_base_url(self) -> str:
        return self.base_url

    def api_call(self, url, api_request_method, query_params=None, request_body=None):
        headers = {
            "Content-Type": "application/json",
            "Authorization": f"Bearer {self.access_token}",
        }
        try:
            if api_request_method == "GET":
                response = requests.get(url, headers=headers, params=query_params)

            elif api_request_method == "DELETE":
                response = requests.delete(url, headers=headers, params=query_params)

            elif api_request_method == "PUT":
                response = requests.put(url, headers=headers, json=request_body)

            else:
                response = requests.post(
                    url, headers=headers, json=request_body, params=query_params
                )

        except Exception as ex:
            raise ValueError(f"Error on calling API {url}:- {ex}")

        return response


In [359]:
import os
from dotenv import load_dotenv

load_dotenv()


def get_cubejs_client():
    return CubeJSClient(
        token=os.getenv("CUBEJS_TOKEN"),
        base_url=os.getenv("AZGEMS_CUBEJS_BASE_URL"),
    )

In [360]:
os.getenv("AZGEMS_CUBEJS_BASE_URL")

'http://0.0.0.0:4000/cubejs-api/v1/load'

In [361]:
shipment_dataset_query_1_cube = """
{{
  "dimensions": [
    "SHIPMENT_ROW_FEATURES.bill_id",
    "SHIPMENT_ROW_FEATURES.bill_id_dup",
    "SHIPMENT_ROW_FEATURES.bni_created_time",
    "SHIPMENT_ROW_FEATURES.coo",
    "SHIPMENT_ROW_FEATURES.days_since_ship_so_far",
    "SHIPMENT_ROW_FEATURES.days_until_eta",
    "SHIPMENT_ROW_FEATURES.delay_days",
    "SHIPMENT_ROW_FEATURES.delivery_terms",
    "SHIPMENT_ROW_FEATURES.eta_dt",
    "SHIPMENT_ROW_FEATURES.item_brand",
    "SHIPMENT_ROW_FEATURES.item_manufacturer",
    "SHIPMENT_ROW_FEATURES.item_product_category",
    "SHIPMENT_ROW_FEATURES.item_size",
    "SHIPMENT_ROW_FEATURES.item_sku",
    "SHIPMENT_ROW_FEATURES.lead_time_days",
    "SHIPMENT_ROW_FEATURES.ocean_freight",
    "SHIPMENT_ROW_FEATURES.po_date_dt",
    "SHIPMENT_ROW_FEATURES.po_shipment_terms",
    "SHIPMENT_ROW_FEATURES.promised_transit_days",
    "SHIPMENT_ROW_FEATURES.quantity_in",
    "SHIPMENT_ROW_FEATURES.receipt_dt",
    "SHIPMENT_ROW_FEATURES.scac",
    "SHIPMENT_ROW_FEATURES.shipment_days",
    "SHIPMENT_ROW_FEATURES.shipped_dt",
    "SHIPMENT_ROW_FEATURES.tariff_amount",
    "SHIPMENT_ROW_FEATURES.tariff_type",
    "SHIPMENT_ROW_FEATURES.total_bcy",
    "SHIPMENT_ROW_FEATURES.vendor_id",
    "SHIPMENT_ROW_FEATURES.vendor_name"
  ],
  "timeDimensions": [],
  "filters": [
    {{
      "values": ["{customer_name}"],
      "member": "SHIPMENT_ROW_FEATURES.customer_name",
      "operator": "contains"
    }}
  ]
}}
""".format(customer_name="Walmart")

shipment_dataset_query_2_cube="""{{
  "dimensions": [
    "SHIPMENT_VENDOR_AGGREGATES.bill_id",
    "SHIPMENT_VENDOR_AGGREGATES.vendor_avg_promised_transit_days",
    "SHIPMENT_VENDOR_AGGREGATES.vendor_avg_realized_delay_days",
    "SHIPMENT_VENDOR_AGGREGATES.vendor_id",
    "SHIPMENT_VENDOR_AGGREGATES.vendor_on_time_rate",
    "SHIPMENT_VENDOR_AGGREGATES.vendor_p50_promised_transit_days",
    "SHIPMENT_VENDOR_AGGREGATES.vendor_p50_realized_delay_days",
    "SHIPMENT_VENDOR_AGGREGATES.vendor_p90_promised_transit_days",
    "SHIPMENT_VENDOR_AGGREGATES.vendor_p90_realized_delay_days",
    "SHIPMENT_VENDOR_AGGREGATES.vendor_shipments_with_receipt"
  ],
  "filters": [
    {{
      "values": [
        "{customer_name}"
      ],
      "member": "SHIPMENT_VENDOR_AGGREGATES.customer_name",
      "operator": "contains"
    }}
  ]
}}
""".format(customer_name="Walmart")


In [362]:
client = get_cubejs_client()

result1 = client.api_call(
                client.get_base_url() + "?query=" + shipment_dataset_query_1_cube, "GET"
            )
dataset_df1 = pd.DataFrame(result1.json()["data"])

# Strip cube name prefix
dataset_df1.columns = [col.split(".")[-1] for col in dataset_df1.columns]



In [363]:

result2 = client.api_call(
                client.get_base_url() + "?query=" + shipment_dataset_query_2_cube, "GET"
            )
print(result2.json())
dataset_df2 = pd.DataFrame(result2.json()["data"])

# Strip cube name prefix
dataset_df2.columns = [col.split(".")[-1] for col in dataset_df2.columns]

{'query': {'dimensions': ['SHIPMENT_VENDOR_AGGREGATES.bill_id', 'SHIPMENT_VENDOR_AGGREGATES.vendor_avg_promised_transit_days', 'SHIPMENT_VENDOR_AGGREGATES.vendor_avg_realized_delay_days', 'SHIPMENT_VENDOR_AGGREGATES.vendor_id', 'SHIPMENT_VENDOR_AGGREGATES.vendor_on_time_rate', 'SHIPMENT_VENDOR_AGGREGATES.vendor_p50_promised_transit_days', 'SHIPMENT_VENDOR_AGGREGATES.vendor_p50_realized_delay_days', 'SHIPMENT_VENDOR_AGGREGATES.vendor_p90_promised_transit_days', 'SHIPMENT_VENDOR_AGGREGATES.vendor_p90_realized_delay_days', 'SHIPMENT_VENDOR_AGGREGATES.vendor_shipments_with_receipt'], 'timeDimensions': [], 'limit': 10000, 'timezone': 'UTC', 'filters': [{'member': 'SHIPMENT_VENDOR_AGGREGATES.customer_name', 'operator': 'contains', 'values': ['Walmart']}], 'rowLimit': 10000}, 'lastRefreshTime': '2025-11-03T08:19:18.475Z', 'refreshKeyValues': [[{'refresh_key': '176215795'}]], 'usedPreAggregations': {}, 'transformedQuery': {'sortedDimensions': ['SHIPMENT_VENDOR_AGGREGATES.bill_id', 'SHIPMENT_VE

In [364]:
dataset_df2.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6427 entries, 0 to 6426
Data columns (total 10 columns):
 #   Column                            Non-Null Count  Dtype 
---  ------                            --------------  ----- 
 0   vendor_p50_promised_transit_days  6427 non-null   object
 1   bill_id                           6427 non-null   object
 2   vendor_p90_realized_delay_days    6427 non-null   object
 3   vendor_shipments_with_receipt     6427 non-null   object
 4   vendor_p90_promised_transit_days  6427 non-null   object
 5   vendor_p50_realized_delay_days    6427 non-null   object
 6   vendor_on_time_rate               6427 non-null   object
 7   vendor_id                         6427 non-null   object
 8   vendor_avg_realized_delay_days    6427 non-null   object
 9   vendor_avg_promised_transit_days  6427 non-null   object
dtypes: object(10)
memory usage: 502.2+ KB


In [365]:
shipment_dataset_df_cube = dataset_df1.merge(dataset_df2, on=["bill_id", "vendor_id"], how="left")

In [366]:
shipment_dataset_df_cube.drop_duplicates(inplace=True)

In [367]:
shipment_dataset_df_cube.dropna(subset=['receipt_dt'], inplace=True)

In [368]:
shipment_dataset_df_cube.info()

<class 'pandas.core.frame.DataFrame'>
Index: 847 entries, 0 to 917
Data columns (total 37 columns):
 #   Column                            Non-Null Count  Dtype 
---  ------                            --------------  ----- 
 0   days_since_ship_so_far            847 non-null    object
 1   quantity_in                       847 non-null    object
 2   item_brand                        847 non-null    object
 3   bill_id                           847 non-null    object
 4   delivery_terms                    847 non-null    object
 5   promised_transit_days             847 non-null    object
 6   item_product_category             847 non-null    object
 7   item_size                         847 non-null    object
 8   shipment_days                     847 non-null    object
 9   vendor_name                       847 non-null    object
 10  po_shipment_terms                 847 non-null    object
 11  scac                              847 non-null    object
 12  lead_time_days             

In [369]:
import pandas as pd
import numpy as np
from pandas.testing import assert_frame_equal
from pandas.api.types import (
    is_numeric_dtype,
    is_datetime64_any_dtype,
    is_timedelta64_dtype,
)

def compare_dfs(df1: pd.DataFrame,
                df2: pd.DataFrame,
                keys=None,
                tol: float = 0.0,
                ignore_order: bool = True,
                ignore_column_order: bool = True):
    report = {}

    report["shape_df1"] = df1.shape
    report["shape_df2"] = df2.shape
    report["columns_df1_only"] = sorted(set(df1.columns) - set(df2.columns))
    report["columns_df2_only"] = sorted(set(df2.columns) - set(df1.columns))

    if keys:
        missing1 = [k for k in keys if k not in df1.columns]
        missing2 = [k for k in keys if k not in df2.columns]
        if missing1 or missing2:
            report["error"] = f"Missing join keys. df1 missing: {missing1}, df2 missing: {missing2}"
            return False, report

        df1a = df1.copy()
        df2a = df2.copy()
        for k in keys:
            df1a[k] = df1a[k].astype(str)
            df2a[k] = df2a[k].astype(str)

        common_cols = sorted(set(df1a.columns).intersection(df2a.columns))
        df1c = df1a[common_cols].drop_duplicates(subset=keys)
        df2c = df2a[common_cols].drop_duplicates(subset=keys)

        df1c = df1c.set_index(keys).sort_index()
        df2c = df2c.set_index(keys).sort_index()
    else:
        common_cols = sorted(set(df1.columns).intersection(df2.columns))
        df1c = df1[common_cols].copy()
        df2c = df2[common_cols].copy()

        if ignore_order:
            df1c = df1c.assign(__row_sort=df1c.astype(str).agg("|".join, axis=1)).sort_values("__row_sort").drop(columns="__row_sort")
            df2c = df2c.assign(__row_sort=df2c.astype(str).agg("|".join, axis=1)).sort_values("__row_sort").drop(columns="__row_sort")

        if ignore_column_order:
            df1c = df1c[sorted(df1c.columns)]
            df2c = df2c[sorted(df2c.columns)]

    # --- FIX: robust casting & datetime normalization ---
    def _normalize_datetime(s: pd.Series) -> pd.Series:
        # Parse to datetime (force UTC), drop tz to compare naive values
        dt = pd.to_datetime(s, errors="coerce", utc=True)
        return dt.dt.tz_convert("UTC").dt.tz_localize(None)

    for col in df1c.columns:
        if is_datetime64_any_dtype(df1c[col]) or is_datetime64_any_dtype(df2c[col]):
            df1c[col] = _normalize_datetime(df1c[col])
            df2c[col] = _normalize_datetime(df2c[col])
        elif is_timedelta64_dtype(df1c[col]) or is_timedelta64_dtype(df2c[col]):
            # leave timedeltas as-is; pandas can compare them
            pass
        elif is_numeric_dtype(df1c[col]) or is_numeric_dtype(df2c[col]):
            df1c[col] = pd.to_numeric(df1c[col], errors="coerce")
            df2c[col] = pd.to_numeric(df2c[col], errors="coerce")
        # else: keep as object/string

    try:
        assert_frame_equal(
            df1c, df2c,
            check_like=ignore_column_order,
            check_exact=(tol == 0.0),
            atol=tol, rtol=0.0,
            check_dtype=False
        )
        report["equal"] = True
        report["note"] = "DataFrames match under the provided settings."
        return True, report
    except AssertionError as e:
        report["equal"] = False
        report["assertion"] = str(e)

    s1 = set(map(tuple, df1c.astype(str).to_numpy()))
    s2 = set(map(tuple, df2c.astype(str).to_numpy()))
    only_in_df1 = s1 - s2
    only_in_df2 = s2 - s1
    report["rows_only_in_df1_count"] = len(only_in_df1)
    report["rows_only_in_df2_count"] = len(only_in_df2)
    if only_in_df1:
        report["sample_rows_only_in_df1"] = [list(r) for r in list(only_in_df1)[:3]]
    if only_in_df2:
        report["sample_rows_only_in_df2"] = [list(r) for r in list(only_in_df2)[:3]]
    return False, report


In [370]:
compare_dfs(shipment_dataset_df, shipment_dataset_df_cube)

(False,
 {'shape_df1': (847, 37),
  'shape_df2': (847, 37),
  'columns_df1_only': [],
  'columns_df2_only': [],
  'equal': False,
  'assertion': "DataFrame.index are different\n\nDataFrame.index values are different (100.0 %)\n[left]:  Index([  52,   76,   92,  108,  124,  140,  142,  156,  196,  198,\n       ...\n       2408, 2410, 2412, 2414, 2415, 2417, 2419, 2420, 2421, 2423],\n      dtype='int64', length=847)\n[right]: Index([  0,   1,   2,   3,   4,   5,   6,   7,   8,   9,\n       ...\n       862, 863, 868, 869, 870, 873, 874, 877, 891, 917],\n      dtype='int64', length=847)",
  'rows_only_in_df1_count': 847,
  'rows_only_in_df2_count': 847,
  'sample_rows_only_in_df1': [['1410873000129078123',
    '1410873000129078123',
    '2025-09-08 17:03:07',
    'INDIA',
    '127.0',
    '-60.0',
    '4',
    'CY',
    '2025-09-04',
    'GREAT VALUE',
    'VANNAMEI',
    'Goods',
    '60/80',
    '870',
    '311.0',
    '5632.0',
    '2024-08-22',
    'DDP',
    '67.0',
    '33750.0',
   

In [371]:
same, info = compare_dfs(shipment_dataset_df, shipment_dataset_df_cube, keys=["bill_id","vendor_id"], tol=0.0)
print(same)
print(info)


False
{'shape_df1': (847, 37), 'shape_df2': (847, 37), 'columns_df1_only': [], 'columns_df2_only': [], 'equal': False, 'assertion': 'DataFrame.iloc[:, 1] (column name="bni_created_time") are different\n\nDataFrame.iloc[:, 1] (column name="bni_created_time") values are different (1.77891 %)\n[index]: [(1410873000050833319, 1410873000047119614), (1410873000050957648, 1410873000014109655), (1410873000050957840, 1410873000014109655), (1410873000050973443, 1410873000014109655), (1410873000050973510, 1410873000014109655), (1410873000053976868, 1410873000014109655), (1410873000054027700, 1410873000047119614), (1410873000054027757, 1410873000047119614), (1410873000054027812, 1410873000014109655), (1410873000054027869, 1410873000014109655), (1410873000054027932, 1410873000000201701), (1410873000054027995, 1410873000000201701), (1410873000054049353, 1410873000000201701), (1410873000054049406, 1410873000000201701), (1410873000054049459, 1410873000000201701), (1410873000054049512, 1410873000000201

In [372]:
inference_cube_query_1 = """ 
{{
  "dimensions": [
    "SHIPMENT_ROW_FEATURES.bill_id",
    "SHIPMENT_ROW_FEATURES.bill_id_dup",
    "SHIPMENT_ROW_FEATURES.bni_created_time",
    "SHIPMENT_ROW_FEATURES.coo",
    "SHIPMENT_ROW_FEATURES.days_since_ship_so_far",
    "SHIPMENT_ROW_FEATURES.days_until_eta",
    "SHIPMENT_ROW_FEATURES.delay_days",
    "SHIPMENT_ROW_FEATURES.delivery_terms",
    "SHIPMENT_ROW_FEATURES.eta_dt",
    "SHIPMENT_ROW_FEATURES.item_brand",
    "SHIPMENT_ROW_FEATURES.item_manufacturer",
    "SHIPMENT_ROW_FEATURES.item_product_category",
    "SHIPMENT_ROW_FEATURES.item_size",
    "SHIPMENT_ROW_FEATURES.item_sku",
    "SHIPMENT_ROW_FEATURES.lead_time_days",
    "SHIPMENT_ROW_FEATURES.ocean_freight",
    "SHIPMENT_ROW_FEATURES.po_date_dt",
    "SHIPMENT_ROW_FEATURES.po_shipment_terms",
    "SHIPMENT_ROW_FEATURES.promised_transit_days",
    "SHIPMENT_ROW_FEATURES.quantity_in",
    "SHIPMENT_ROW_FEATURES.receipt_dt",
    "SHIPMENT_ROW_FEATURES.scac",
    "SHIPMENT_ROW_FEATURES.shipment_days",
    "SHIPMENT_ROW_FEATURES.shipped_dt",
    "SHIPMENT_ROW_FEATURES.tariff_amount",
    "SHIPMENT_ROW_FEATURES.tariff_type",
    "SHIPMENT_ROW_FEATURES.total_bcy",
    "SHIPMENT_ROW_FEATURES.vendor_id",
    "SHIPMENT_ROW_FEATURES.vendor_name"
  ],
  "timeDimensions": [],
  "filters": [
    {{
      "values": [
        "{customer_name}"
      ],
      "member": "SHIPMENT_ROW_FEATURES.customer_name",
      "operator": "contains"
}},
    {{
      "values": [
        "{start_timestamp}",
        "{end_timestamp}"
      ],
      "member": "SHIPMENT_ROW_FEATURES.bni_created_time",
      "operator": "inDateRange"
    }}
  ]
}}
""".format(
    customer_name="Walmart", start_timestamp="2025-08-01", end_timestamp="2025-09-08"
)

inference_cube_query_2 = """ {{
  "dimensions": [
    "SHIPMENT_VENDOR_AGGREGATES.bill_id",
    "SHIPMENT_VENDOR_AGGREGATES.customer_name",
    "SHIPMENT_VENDOR_AGGREGATES.vendor_avg_promised_transit_days",
    "SHIPMENT_VENDOR_AGGREGATES.vendor_avg_realized_delay_days",
    "SHIPMENT_VENDOR_AGGREGATES.vendor_id",
    "SHIPMENT_VENDOR_AGGREGATES.vendor_on_time_rate",
    "SHIPMENT_VENDOR_AGGREGATES.vendor_p50_promised_transit_days",
    "SHIPMENT_VENDOR_AGGREGATES.vendor_p50_realized_delay_days",
    "SHIPMENT_VENDOR_AGGREGATES.vendor_p90_promised_transit_days",
    "SHIPMENT_VENDOR_AGGREGATES.vendor_p90_realized_delay_days",
    "SHIPMENT_VENDOR_AGGREGATES.vendor_shipments_with_receipt"
  ],
  "filters": [
    {{
      "values": [
        "{customer_name}"
      ],
      "member": "SHIPMENT_VENDOR_AGGREGATES.customer_name",
      "operator": "contains"
    }}
  ]
}}""".format(
    customer_name="Walmart"
)

In [373]:
inference_result = client.api_call(
                client.get_base_url() + "?query=" + inference_cube_query_1, "GET"
            )
inference_dataset_df1 = pd.DataFrame(inference_result.json()["data"])

# Strip cube name prefix
inference_dataset_df1.columns = [col.split(".")[-1] for col in inference_dataset_df1.columns]

In [374]:
inference_result2 = client.api_call(
                client.get_base_url() + "?query=" + inference_cube_query_2, "GET"
            )
print(inference_result2)
inference_dataset_df2 = pd.DataFrame(inference_result2.json()["data"])

# Strip cube name prefix
inference_dataset_df2.columns = [col.split(".")[-1] for col in inference_dataset_df2.columns]

<Response [200]>


In [375]:
inference_dataset = inference_dataset_df1.merge(inference_dataset_df2, on=["bill_id", "vendor_id"], how="left")

In [376]:
inference_dataset.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 52 entries, 0 to 51
Data columns (total 38 columns):
 #   Column                            Non-Null Count  Dtype 
---  ------                            --------------  ----- 
 0   quantity_in                       52 non-null     object
 1   item_product_category             52 non-null     object
 2   days_since_ship_so_far            52 non-null     object
 3   eta_dt                            52 non-null     object
 4   lead_time_days                    52 non-null     object
 5   scac                              52 non-null     object
 6   vendor_name                       52 non-null     object
 7   tariff_type                       52 non-null     object
 8   coo                               52 non-null     object
 9   delivery_terms                    52 non-null     object
 10  shipped_dt                        52 non-null     object
 11  tariff_amount                     42 non-null     object
 12  item_brand              

In [377]:
inference_dataset.drop_duplicates()

,quantity_in,item_product_category,days_since_ship_so_far,eta_dt,lead_time_days,scac,vendor_name,tariff_type,coo,delivery_terms,...,receipt_dt,vendor_shipments_with_receipt,vendor_avg_promised_transit_days,vendor_p90_realized_delay_days,vendor_on_time_rate,vendor_avg_realized_delay_days,vendor_p90_promised_transit_days,vendor_p50_promised_transit_days,vendor_p50_realized_delay_days,customer_name
0,31200,Goods,170,2025-07-02T00:00:00.000,37,CMDU,Sandhya Aqua Exports Pvt Ltd,Price Without Tariff,INDIA,CY,...,2025-08-02T00:00:00.000,214,49.95,6,0,3.52,57,49,3,Walmart
1,31752,Goods,164,2025-07-10T00:00:00.000,21,CMDU,Aquatica Frozen Foods Global Pvt Ltd,Price With Tariff,INDIA,CY,...,2025-07-15T00:00:00.000,54,52.8,6,0,4.91,72,49,4,Walmart
2,33600,Goods,155,2025-08-08T00:00:00.000,283,MAEU,Neeli Aqua,Price With Tariff,INDIA,CY,...,2025-08-11T00:00:00.000,14,62.93,6,0,4.64,74,60,5,Walmart
3,35200,Goods,146,2025-07-30T00:00:00.000,41,CMDU,Sandhya Aqua Exports Pvt Ltd,Price Without Tariff,INDIA,CY,...,2025-08-04T00:00:00.000,212,49.8,6,0,3.77,57,49,3,Walmart
4,35280,Goods,146,2025-07-30T00:00:00.000,77,CMDU,Sandhya Aqua Exports Pvt Ltd,,INDIA,CY,...,2025-08-01T00:00:00.000,212,49.8,6,0,3.77,57,49,3,Walmart
5,16848,Goods,146,2025-07-28T00:00:00.000,75,CMDU,Sandhya Aqua Exports Pvt Ltd,Price Without Tariff,INDIA,CY,...,2025-08-01T00:00:00.000,212,49.8,6,0,3.77,57,49,3,Walmart
6,12,Goods,147,2025-07-21T00:00:00.000,90,COSU,Anova,,VIETNAM,Door Delivered,...,2025-07-24T00:00:00.000,4,55,4,0,3.25,77,49,4,Walmart
7,12,Goods,147,2025-07-21T00:00:00.000,90,COSU,Anova,Price With Tariff,VIETNAM,Door Delivered,...,2025-07-24T00:00:00.000,4,55,4,0,3.25,77,49,4,Walmart
8,36,Goods,142,2025-08-06T00:00:00.000,66,CMDU,Sandhya Aqua Exports Pvt Ltd,Price Without Tariff,INDIA,CY,...,2025-08-07T00:00:00.000,219,49.79,6,0,3.74,57,49,3,Walmart
9,33984,Goods,142,2025-08-06T00:00:00.000,66,CMDU,Sandhya Aqua Exports Pvt Ltd,Price Without Tariff,INDIA,CY,...,2025-08-07T00:00:00.000,219,49.79,6,0,3.74,57,49,3,Walmart


In [378]:
string = "SHIPMENT_ROW_FEATURES"
string.upper()

'SHIPMENT_ROW_FEATURES'